# Week 6: FinBERT Retraining on Real Data

This notebook fine-tunes the `FinBertFusionClassifier` on the real financial corpus (`data/financial_corpus.csv`) generated in Week 6.

In [ ]:
!pip install -q torch transformers datasets scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pickle
import os

In [ ]:
# 1. Load Data
df = pd.read_csv('../data/financial_corpus.csv')
print(f"Total records: {len(df)}")
df.head(3)

In [ ]:
# Encode labels
# UP = 2, DOWN = 0, HOLD = 1
label_map = {'DOWN': 0, 'HOLD': 1, 'UP': 2}
df['label_id'] = df['label'].map(label_map)

# Split 80/20 stratified
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['label_id'], 
    random_state=42
)
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}")

In [ ]:
# 2. Dataset and Tokenizer
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

class FinBERTDataset(Dataset):
    def __init__(self, texts, price_features, labels, tokenizer, max_length=128):
        self.texts = texts
        self.price_features = price_features
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        pf = self.price_features[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'price_features': torch.tensor(pf, dtype=torch.float),
            'label': torch.tensor(label, dtype=torch.long)
        }

train_pf = train_df[['price_5d_return', 'volume_change_pct']].fillna(0.0).values
val_pf = val_df[['price_5d_return', 'volume_change_pct']].fillna(0.0).values

train_dataset = FinBERTDataset(train_df['cleaned_text'].values, train_pf, train_df['label_id'].values, tokenizer)
val_dataset = FinBERTDataset(val_df['cleaned_text'].values, val_pf, val_df['label_id'].values, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [ ]:
# 3. Model Architecture
class FinBertFusionClassifier(nn.Module):
    def __init__(self, freeze_bert=True):
        super(FinBertFusionClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained("ProsusAI/finbert")
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
                
        self.fusion_layer = nn.Linear(768 + 2, 128)
        self.output_layer = nn.Linear(128, 3)
        self.relu = nn.ReLU()
        # Note: We output raw logits for CrossEntropyLoss

    def forward(self, input_ids, attention_mask, price_features):
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vectors = bert_outputs.last_hidden_state[:, 0, :]
        fused_vectors = torch.cat((cls_vectors, price_features), dim=1)
        x = self.relu(self.fusion_layer(fused_vectors))
        logits = self.output_layer(x)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FinBertFusionClassifier(freeze_bert=True).to(device)
print(f"Using device: {device}")

In [ ]:
# 4. Training Loop
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)
criterion = nn.CrossEntropyLoss()

epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        price_features = batch['price_features'].to(device)
        labels = batch['label'].to(device)
        
        logits = model(input_ids, attention_mask, price_features)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    model.eval()
    val_preds = []
    val_labels = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            price_features = batch['price_features'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask, price_features)
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
            
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f} - Val Acc: {val_acc:.4f}")

In [ ]:
# 5. Evaluation and Comparison
import sys
sys.path.append('../src')
try:
    from forecast_model import forecast_from_news
except ImportError:
    forecast_from_news = None

# Evaluate FinBERT
precision, recall, f1, _ = precision_recall_fscore_support(val_labels, val_preds, average='weighted', zero_division=0)
print(f"FinBERT - Acc: {val_acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

# Evaluate Rule-based Baseline
rb_preds = []
if forecast_from_news:
    for _, row in val_df.iterrows():
        news_items = [{'cleaned_text': row['cleaned_text']}]
        price_feats = {'price_5d_return': row['price_5d_return'], 'volume_change_pct': row['volume_change_pct']}
        rb_res = forecast_from_news(news_items, price_feats)
        rb_label = label_map[rb_res['prediction']]
        rb_preds.append(rb_label)
        
    rb_acc = accuracy_score(val_labels, rb_preds)
    print(f"Rule-based Baseline - Acc: {rb_acc:.4f}")
    
    print("\nComparison Table (Sample):")
    comp_df = pd.DataFrame({
        'True Label': val_labels,
        'Rule-Based Pred': rb_preds,
        'FinBERT Pred': val_preds
    }).head(10)
    print(comp_df)

In [ ]:
# 6. Export Model
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/finbert_fusion.pt')
with open('../models/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_map, f)
print("Model exported to models/finbert_fusion.pt")